# Classification: Temporal Features

Instead of averaging metrics across all trials, each session is split into the first third (early) and the last third (late) of trials. For each segment we compute the per-session mean of each metric, then add delta features (late − early) to capture within-session change - attention deterioration, fatigue, habituation. After computing per-session early / late values, features are averaged per user and deltas are recomputed at the user level to reduce within-subject measurement noise.

Three tasks are tested:
- binary classification (depressed vs not)
- multi-class (severity groups)
- regression (predict score directly)

Every task is run on two depression scales:
- **PHQ-9** (Patient Health Questionnaire, 9 items, range 0–27)
- **BDI-II** (Beck Depression Inventory, 21 items, range 0–63)

Running both scales provides a convergent-validity check: if the same gaze features predict both questionnaires similarly, the signal reflects the depression construct rather than questionnaire-specific content.

Three models are compared:
- Logistic / Ridge Regression
- Random Forest
- XGBoost

Across three feature sets:
- all temporal (early + late + delta for every base metric)
- theory-driven temporal (early + late + delta only on Beck-theory metrics)
- delta only (within-session change only; tests whether change carries incremental signal over level)

Features are averaged per user before modeling, so every user appears in exactly one fold. Cross-validation uses 5-fold GroupKFold by `uid`, which at this resolution reduces to standard 5-fold KFold.

In [0]:
%pip install xgboost -q

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import pandas as pd
import numpy as np

from src.classification import (
    prepare_data, run_classification_binary, run_classification_multiclass, run_regression,
    plot_best_classification_binary, plot_best_classification_multiclass, plot_best_regression,
    plot_summary, plot_feature_importance,
)

## 1. Build session-level dataset

### 1.1 Build early / late session features

In [0]:
scene_metrics = spark.table("anima.scene_metrics")
df_forms = spark.table("anima.forms")

df_stimulus = scene_metrics.filter(F.col("scene_type") == "stimulus")

BASE_METRICS = [
    "fixation_count",
    "mean_fixation_duration_ms",
    "total_fixation_duration_ms",
    "fixation_rate_per_sec",
    "fixation_bias",
    "scanpath_length",
    "saccade_count",
    "saccade_rate_per_sec",
    "mean_saccade_amplitude",
    "blink_count",
    "blink_rate_per_min",
    "transition_matrix_density",
    "gaze_transition_entropy",
    "first_fixation_duration_ms",
    "second_fixation_duration_ms",
    "dwell_time_ms_negative",
    "dwell_time_ms_positive",
    "dwell_time_ms_neutral",
    "dwell_time_500ms_negative",
    "dwell_time_500ms_positive",
    "dwell_time_500ms_neutral",
    "fixation_proportion_negative",
    "fixation_proportion_positive",
    "fixation_proportion_neutral",
    "fixation_count_negative",
    "fixation_count_positive",
    "fixation_count_neutral",
    "revisit_count_negative",
    "revisit_count_positive",
    "revisit_count_neutral",
    "ttff_ms_negative",
    "ttff_ms_positive",
    "ttff_ms_neutral",
]

w = Window.partitionBy("session_id").orderBy("scene_index")
numbered = df_stimulus.withColumn("trial_num", F.row_number().over(w))
numbered = numbered.withColumn("n_trials", F.count("*").over(Window.partitionBy("session_id")))

early = numbered.filter(F.col("trial_num") <= F.col("n_trials") / 3)
late = numbered.filter(F.col("trial_num") > F.col("n_trials") * 2 / 3)

def build_agg(suffix):
    agg_exprs = []
    for m in BASE_METRICS:
        agg_exprs.append(F.mean(m).alias(f"{m}_{suffix}"))
    for valence in ["negative", "positive", "neutral"]:
        agg_exprs.append(
            F.avg(F.when(F.col("first_fixation_valence") == valence, 1).otherwise(0))
             .alias(f"first_fix_prob_{valence}_{suffix}"))
        agg_exprs.append(
            F.avg(F.when(F.col("second_fixation_valence") == valence, 1).otherwise(0))
             .alias(f"second_fix_prob_{valence}_{suffix}"))
    return agg_exprs

early_metrics = early.groupBy("session_id").agg(*build_agg("early"))
late_metrics = late.groupBy("session_id").agg(*build_agg("late"))

session_metrics = early_metrics.join(late_metrics, on="session_id", how="inner")

df_joined = session_metrics.join(
    df_forms.select("session_id", "uid", "phq9_score", "bdi_score"),
    on="session_id", how="inner",
)

df = df_joined.toPandas()

print(f"Sessions: {len(df)}, Users: {df['uid'].nunique()}")

### 1.2 Aggregate to user level and compute deltas

Features are averaged per user across their sessions. Delta features (late − early) are computed at the user level from the aggregated values, so each user has one early mean, one late mean, and one delta per metric.

In [0]:
FEATURE_COLS = [c for c in df.columns if c.endswith(("_early", "_late"))]
NUMERIC_COLS = FEATURE_COLS + ["phq9_score", "bdi_score"]

df = df.groupby("uid", as_index=False)[NUMERIC_COLS].mean()

# Compute delta features at user level: late - early
for m in BASE_METRICS:
    df[f"{m}_delta"] = df[f"{m}_late"] - df[f"{m}_early"]
for valence in ["negative", "positive", "neutral"]:
    df[f"first_fix_prob_{valence}_delta"] = df[f"first_fix_prob_{valence}_late"] - df[f"first_fix_prob_{valence}_early"]
    df[f"second_fix_prob_{valence}_delta"] = df[f"second_fix_prob_{valence}_late"] - df[f"second_fix_prob_{valence}_early"]

print(f"After per-user aggregation: {len(df)} users")
print(f"PHQ-9: min={df['phq9_score'].min():.1f}, max={df['phq9_score'].max():.1f}, mean={df['phq9_score'].mean():.1f}")
print(f"BDI-II: min={df['bdi_score'].min():.1f}, max={df['bdi_score'].max():.1f}, mean={df['bdi_score'].mean():.1f}")

## 2. Define targets and feature sets

### 2.1 PHQ-9 targets

Binary cutoff: PHQ-9 ≥ 10 (standard Kroenke clinical threshold).
Severity classes (Kroenke et al., 2001):
- 0 = minimal (0–4)
- 1 = mild (5–9)
- 2 = moderate (10–14)
- 3 = moderately severe (15–19)
- 4 = severe (20–27)

In [0]:
df["phq9_depressed"] = (df["phq9_score"] >= 10).astype(int)

def phq9_severity_from_score(s):
    if s <= 4:  return 0   # minimal
    if s <= 9:  return 1   # mild
    if s <= 14: return 2   # moderate
    if s <= 19: return 3   # moderately severe
    return 4               # severe

df["phq9_severity_class"] = df["phq9_score"].apply(phq9_severity_from_score)

print("PHQ-9")
print(f"  Binary (>=10): {df['phq9_depressed'].value_counts().to_dict()}")
print(f"  Multi-class:   {df['phq9_severity_class'].value_counts().sort_index().to_dict()}")

### 2.2 BDI-II targets

Binary cutoff: BDI-II ≥ 14 (Beck et al., 1996 threshold for mild or worse depression; 81% sensitivity, 92% specificity against DSM-IV MDD).
Severity classes (Beck et al., 1996):
- 0 = minimal (0–13)
- 1 = mild (14–19)
- 2 = moderate (20–28)
- 3 = severe (29–63)

In [0]:
df["bdi_depressed"] = (df["bdi_score"] >= 14).astype(int)

def bdi_severity_from_score(s):
    if s <= 13: return 0   # minimal
    if s <= 19: return 1   # mild
    if s <= 28: return 2   # moderate
    return 3               # severe

df["bdi_severity_class"] = df["bdi_score"].apply(bdi_severity_from_score)

print("BDI-II")
print(f"  Binary (>=14): {df['bdi_depressed'].value_counts().to_dict()}")
print(f"  Multi-class:   {df['bdi_severity_class'].value_counts().sort_index().to_dict()}")

### 2.3 Feature sets

In [0]:
EARLY_FEATURES = [f"{m}_early" for m in BASE_METRICS]
LATE_FEATURES = [f"{m}_late" for m in BASE_METRICS]
DELTA_FEATURES = [f"{m}_delta" for m in BASE_METRICS]

PROB_EARLY = [f"{p}_{v}_early" for p in ["first_fix_prob", "second_fix_prob"] for v in ["negative", "positive", "neutral"]]
PROB_LATE  = [f"{p}_{v}_late"  for p in ["first_fix_prob", "second_fix_prob"] for v in ["negative", "positive", "neutral"]]
PROB_DELTA = [f"{p}_{v}_delta" for p in ["first_fix_prob", "second_fix_prob"] for v in ["negative", "positive", "neutral"]]

ALL_FEATURES = EARLY_FEATURES + LATE_FEATURES + DELTA_FEATURES + PROB_EARLY + PROB_LATE + PROB_DELTA

THEORY_BASE = [
    "fixation_bias", "dwell_time_ms_negative", "dwell_time_ms_positive",
    "fixation_proportion_negative", "fixation_proportion_positive",
    "revisit_count_negative", "revisit_count_positive",
    "blink_rate_per_min", "scanpath_length",
]
THEORY_FEATURES = []
for m in THEORY_BASE:
    THEORY_FEATURES.extend([f"{m}_early", f"{m}_late", f"{m}_delta"])
THEORY_FEATURES.extend([f"first_fix_prob_{v}_{s}" for v in ["negative", "positive"] for s in ["early", "late", "delta"]])

DELTA_ONLY_FEATURES = DELTA_FEATURES + PROB_DELTA

FEATURE_SETS = {
    "All Temporal": ALL_FEATURES,
    "Theory-Driven Temporal": THEORY_FEATURES,
    "Delta Only": DELTA_ONLY_FEATURES,
}

for name, feats in FEATURE_SETS.items():
    print(f"{name}: {len(feats)} features")

## 3. Prepare data

In [0]:
target_cols = [
    "phq9_score", "phq9_depressed", "phq9_severity_class",
    "bdi_score",  "bdi_depressed",  "bdi_severity_class",
]
df_clean, groups = prepare_data(df, FEATURE_SETS, target_cols)

## 4. Binary classification

Four binary analyses crossing two depression scales with two thresholding strategies:
- clinical cutoff (PHQ-9 ≥ 10, BDI-II ≥ 14)
- extremes task (drop the ambiguous middle; compare minimal vs severe)


### 4.1 PHQ-9 cutoff (≥ 10)

#### 4.1.1 Run classification

In [0]:
y_phq9_binary = df_clean["phq9_depressed"].values
phq9_binary_df = run_classification_binary(df_clean, FEATURE_SETS, y_phq9_binary, groups)

#### 4.1.2 Results

In [0]:
print(phq9_binary_df.to_string(index=False))

#### 4.1.3 Best model

In [0]:
plot_best_classification_binary(df_clean, FEATURE_SETS, y_phq9_binary, groups, phq9_binary_df)

### 4.2 PHQ-9 extremes (minimal vs severe)

Minimal (PHQ-9 ≤ 4) vs Severe (PHQ-9 ≥ 20). Users with intermediate scores are excluded.

#### 4.2.1 Run classification

In [0]:
df_phq9_extreme = df_clean[(df_clean["phq9_score"] <= 4) | (df_clean["phq9_score"] >= 20)].copy()
df_phq9_extreme["phq9_extreme"] = (df_phq9_extreme["phq9_score"] >= 20).astype(int)

groups_phq9_extreme = df_phq9_extreme["uid"].values

print(f"PHQ-9 extremes dataset: {len(df_phq9_extreme)} users")
print(f"  Minimal (<=4):  {(df_phq9_extreme['phq9_extreme'] == 0).sum()}")
print(f"  Severe  (>=20): {(df_phq9_extreme['phq9_extreme'] == 1).sum()}")

y_phq9_extreme = df_phq9_extreme["phq9_extreme"].values
phq9_extreme_df = run_classification_binary(df_phq9_extreme, FEATURE_SETS, y_phq9_extreme, groups_phq9_extreme)

#### 4.2.2 Results

In [0]:
print(phq9_extreme_df.to_string(index=False))

#### 4.2.3 Best model

In [0]:
plot_best_classification_binary(df_phq9_extreme, FEATURE_SETS, y_phq9_extreme, groups_phq9_extreme, phq9_extreme_df)

### 4.3 BDI-II cutoff (≥ 14)

#### 4.3.1 Run classification

In [0]:
y_bdi_binary = df_clean["bdi_depressed"].values
bdi_binary_df = run_classification_binary(df_clean, FEATURE_SETS, y_bdi_binary, groups)

#### 4.3.2 Results

In [0]:
print(bdi_binary_df.to_string(index=False))

#### 4.3.3 Best model

In [0]:
plot_best_classification_binary(df_clean, FEATURE_SETS, y_bdi_binary, groups, bdi_binary_df)

### 4.4 BDI-II extremes (minimal vs severe)

Minimal (BDI ≤ 13) vs Severe (BDI ≥ 29). Users with intermediate scores are excluded.

#### 4.4.1 Run classification

In [0]:
df_bdi_extreme = df_clean[(df_clean["bdi_score"] <= 13) | (df_clean["bdi_score"] >= 29)].copy()
df_bdi_extreme["bdi_extreme"] = (df_bdi_extreme["bdi_score"] >= 29).astype(int)

groups_bdi_extreme = df_bdi_extreme["uid"].values

print(f"BDI-II extremes dataset: {len(df_bdi_extreme)} users")
print(f"  Minimal (<=13): {(df_bdi_extreme['bdi_extreme'] == 0).sum()}")
print(f"  Severe  (>=29): {(df_bdi_extreme['bdi_extreme'] == 1).sum()}")

y_bdi_extreme = df_bdi_extreme["bdi_extreme"].values
bdi_extreme_df = run_classification_binary(df_bdi_extreme, FEATURE_SETS, y_bdi_extreme, groups_bdi_extreme)

#### 4.4.2 Results

In [0]:
print(bdi_extreme_df.to_string(index=False))

#### 4.4.3 Best model

In [0]:
plot_best_classification_binary(df_bdi_extreme, FEATURE_SETS, y_bdi_extreme, groups_bdi_extreme, bdi_extreme_df)

## 5. Multi-class classification

### 5.1 PHQ-9 (5 severity classes)

#### 5.1.1 Run classification

In [0]:
PHQ9_LABELS = ["Minimal", "Mild", "Moderate", "Moderately Severe", "Severe"]
y_phq9_multi = df_clean["phq9_severity_class"].values.astype(int)
phq9_multi_df = run_classification_multiclass(df_clean, FEATURE_SETS, y_phq9_multi, groups)

#### 5.1.2 Results

In [0]:
print(phq9_multi_df.to_string(index=False))

#### 5.1.3 Best model

In [0]:
plot_best_classification_multiclass(df_clean, FEATURE_SETS, y_phq9_multi, groups, phq9_multi_df, PHQ9_LABELS)

### 5.2 BDI-II (4 severity classes)

#### 5.2.1 Run classification

In [0]:
BDI_LABELS = ["Minimal", "Mild", "Moderate", "Severe"]
y_bdi_multi = df_clean["bdi_severity_class"].values.astype(int)
bdi_multi_df = run_classification_multiclass(df_clean, FEATURE_SETS, y_bdi_multi, groups)

#### 5.2.2 Results

In [0]:
print(bdi_multi_df.to_string(index=False))

#### 5.2.3 Best model

In [0]:
plot_best_classification_multiclass(df_clean, FEATURE_SETS, y_bdi_multi, groups, bdi_multi_df, BDI_LABELS)

## 6. Regression

### 6.1 PHQ-9 score prediction

#### 6.1.1 Run regression

In [0]:
y_phq9_reg = df_clean["phq9_score"].values
phq9_reg_df = run_regression(df_clean, FEATURE_SETS, y_phq9_reg, groups)

#### 6.1.2 Results

In [0]:
print(phq9_reg_df.to_string(index=False))

#### 6.1.3 Best model

In [0]:
plot_best_regression(df_clean, FEATURE_SETS, y_phq9_reg, groups, phq9_reg_df, score_name="PHQ-9")

### 6.2 BDI-II score prediction

#### 6.2.1 Run regression

In [0]:
y_bdi_reg = df_clean["bdi_score"].values
bdi_reg_df = run_regression(df_clean, FEATURE_SETS, y_bdi_reg, groups)

#### 6.2.2 Results

In [0]:
print(bdi_reg_df.to_string(index=False))

#### 6.2.3 Best model

In [0]:
plot_best_regression(df_clean, FEATURE_SETS, y_bdi_reg, groups, bdi_reg_df, score_name="BDI-II")

## 7. PHQ-9 vs BDI-II convergent-validity comparison

Side-by-side best performance across all tasks. If the two scales produce similar numbers, the gaze-based depression signal generalizes across questionnaires.

In [0]:
def _best_auc(df_res):
    return df_res.sort_values("auc_roc", ascending=False).iloc[0]

def _best_f1(df_res):
    return df_res.sort_values("f1_weighted", ascending=False).iloc[0]

def _best_r2(df_res):
    return df_res.sort_values("r2", ascending=False).iloc[0]

rows = []

# Binary cutoff
p = _best_auc(phq9_binary_df)
b = _best_auc(bdi_binary_df)
rows.append({"task": "Binary cutoff",
             "phq9_metric": "AUC",  "phq9_value": p["auc_roc"],
             "phq9_model": f"{p['model']} / {p['feature_set']}",
             "bdi_metric": "AUC",   "bdi_value": b["auc_roc"],
             "bdi_model": f"{b['model']} / {b['feature_set']}"})

# Binary extremes
p = _best_auc(phq9_extreme_df)
b = _best_auc(bdi_extreme_df)
rows.append({"task": "Binary extremes",
             "phq9_metric": "AUC",  "phq9_value": p["auc_roc"],
             "phq9_model": f"{p['model']} / {p['feature_set']}",
             "bdi_metric": "AUC",   "bdi_value": b["auc_roc"],
             "bdi_model": f"{b['model']} / {b['feature_set']}"})

# Multi-class
p = _best_f1(phq9_multi_df)
b = _best_f1(bdi_multi_df)
rows.append({"task": "Multi-class",
             "phq9_metric": "F1",   "phq9_value": p["f1_weighted"],
             "phq9_model": f"{p['model']} / {p['feature_set']}",
             "bdi_metric": "F1",    "bdi_value": b["f1_weighted"],
             "bdi_model": f"{b['model']} / {b['feature_set']}"})

# Regression
p = _best_r2(phq9_reg_df)
b = _best_r2(bdi_reg_df)
rows.append({"task": "Regression",
             "phq9_metric": "R2",   "phq9_value": p["r2"],
             "phq9_model": f"{p['model']} / {p['feature_set']}",
             "bdi_metric": "R2",    "bdi_value": b["r2"],
             "bdi_model": f"{b['model']} / {b['feature_set']}"})

comparison_df = pd.DataFrame(rows)
print(comparison_df.to_string(index=False))

## 8. Feature importance

Feature importance on the PHQ-9 binary task (Random Forest, all temporal features). Presented once because PHQ-9 and BDI-II converge on similar predictors.

In [0]:
plot_feature_importance(df_clean, ALL_FEATURES, y_phq9_binary, title="Feature importance (PHQ-9 binary, all temporal)")

## 9. Summary

### 9.1 PHQ-9

In [0]:
feature_order = list(FEATURE_SETS.keys())
plot_summary(phq9_binary_df, phq9_multi_df, phq9_reg_df, feature_order, title="PHQ-9 — temporal features")

### 9.2 BDI-II

In [0]:
plot_summary(bdi_binary_df, bdi_multi_df, bdi_reg_df, feature_order, title="BDI-II — temporal features")